In [1]:
from datatypes import RobotCalibration, StrokeSequence, StrokePath
from robot import SerialPrinter, Printer

In [2]:
printer = SerialPrinter()
printer.connect()

Connecting to GRBL on 0...
Connecting to GRBL on 1...
Connecting to GRBL on 2...
Connecting to GRBL on 3...
Connecting to GRBL on 4...
Connecting to GRBL on 5...
Waiting...
Connected successfully.


In [3]:
import math

def load_brush(printer: Printer, my_robot_calibration, color_index):
    """
    1. Navigates to the water cup and agitates to wet the brush.
    2. Navigates to a color palette position and swirls to load paint.
    
    :param printer: An instance of the SerialPrinter/Printer class
    :param palette_position: A tuple of (X, Y, Z) for the specific paint well
    """
    Z_MARGIN = 5
    FEED_RATE_TRAVEL = 2000
    FEED_RATE_WET = 1000
    FEED_RATE_LOAD = 500

    x_paint, y_paint, z_paint = my_robot_calibration.color_palette.color_positions[color_index]["position"]
    x_water, y_water, z_water = my_robot_calibration.water_reservoir

    print("--- Starting Full Brush Prep Sequence ---")

    # ==========================================
    # PHASE 1: WET THE BRUSH IN WATER
    # ==========================================
    print(f"Moving to water reservoir at ({x_water}, {y_water})...")
    # Lift to safe height, move to water cup, and dip down
    printer.move_to(z=z_water+Z_MARGIN, feed_rate=FEED_RATE_TRAVEL)
    printer.move_to(x=x_water, y=y_water, feed_rate=FEED_RATE_TRAVEL)
    printer.move_to(z=z_water, feed_rate=FEED_RATE_TRAVEL)

    print("Agitating brush in water...")
    # Perform a rapid mechanical "shake" to flex bristles and soak up water
    shake_distance = 4.0  # mm
    for _ in range(3):
        # Shake left/right, up/down relative to the center of the water cup
        printer.move_to(x=x_water + shake_distance, y=y_water, feed_rate=FEED_RATE_WET)
        printer.move_to(x=x_water - shake_distance, y=y_water, feed_rate=FEED_RATE_WET)
        printer.move_to(x=x_water, y=y_water + shake_distance, feed_rate=FEED_RATE_WET)
        printer.move_to(x=x_water, y=y_water - shake_distance, feed_rate=FEED_RATE_WET)
    
    # Return cleanly to center of water cup and lift up
    printer.move_to(x=x_water, y=y_water, feed_rate=FEED_RATE_WET)
    printer.move_to(z=z_water+Z_MARGIN, feed_rate=FEED_RATE_TRAVEL)


    # ==========================================
    # PHASE 2: LOAD THE PAINT
    # ==========================================
    print(f"Moving to paint palette at ({x_paint}, {y_paint})...")
    # Move over the target well, dip down to paint height
    printer.move_to(x=x_paint, y=y_paint, feed_rate=FEED_RATE_TRAVEL)
    printer.move_to(z=z_paint, feed_rate=FEED_RATE_TRAVEL)

    print("Swirling brush to load paint...")
    circle_radius = 5.0  # mm
    steps = 16
    for i in range(steps + 1):
        angle = (2 * math.pi / steps) * i
        circle_x = x_paint + circle_radius * math.cos(angle)
        circle_y = y_paint + circle_radius * math.sin(angle)
        printer.move_to(x=circle_x, y=circle_y, feed_rate=FEED_RATE_LOAD)

    # Move back up to clear the well completely before drawing or traveling
    printer.move_to(z=z_paint+Z_MARGIN, feed_rate=FEED_RATE_TRAVEL)
    
    print("--- Brush prep complete and ready to paint! ---")

In [4]:
# --- Physical Painting Heights ---
Z_MARGIN = 5.0         # Safe Z height to clear canvas during travel
FEED_RATE_TRAVEL = 2000 # Fast travel speed
FEED_RATE_PAINT = 1200   # Controlled painting speed


def execute_stroke(printer: Printer, robot_calibration:RobotCalibration, stroke_sequence: StrokeSequence, index: int) -> None:
    """
    Fetches a specific stroke path by index from a StrokeSequence, lifts the brush,
    travels to the starting position, drops down, and traces the coordinates.
    
    :param printer: The connected Printer instance (e.g., SerialPrinter)
    :param stroke_sequence: The StrokeSequence object containing the stroke list
    :param index: Index of the stroke to execute
    """
    up_heigth = robot_calibration.bottom_left[2] + Z_MARGIN
    down_height = robot_calibration.bottom_left[2]
    
    # 1. Bounds check to ensure the index exists
    if index < 0 or index >= len(stroke_sequence.strokes):
        print(f"Error: Stroke index {index} out of bounds (0 to {len(stroke_sequence.strokes)-1}).")
        return

    # 2. Extract the specific stroke data
    stroke: StrokePath = stroke_sequence.strokes[index]
    
    # Safety check: ensure the stroke path actually has points
    if not stroke.path:
        print(f"Stroke at index {index} has an empty path. Skipping.")
        return

    print(f"--- Executing Stroke {index} | Color: {stroke.color} | Width: {stroke.brushWidth} ---")
    print(f"Path {stroke.path}")
    
    # 3. Pull the starting coordinate
    start_x, start_y = stroke.path[0]
    start_x += robot_calibration.bottom_left[0]
    start_y += robot_calibration.bottom_left[1]

    # 4. Lift up to travel height first (prevent dragging across previous paint)
    printer.move_to(z=up_heigth, feed_rate=FEED_RATE_TRAVEL)

    # 5. Travel horizontally to the start position of the stroke
    printer.move_to(x=start_x, y=start_y, feed_rate=FEED_RATE_TRAVEL)

    # 6. Lower the brush onto the canvas
    printer.move_to(z=down_height, feed_rate=FEED_RATE_TRAVEL)

    # 7. Trace out the rest of the points on the canvas at painting speed
    # (Starting from index 1 because we are already at point 0)
    for next_point in stroke.path[1:]:
        next_x, next_y = next_point
        printer.move_to(x=next_x, y=next_y, feed_rate=FEED_RATE_PAINT)

    # 8. Lift the brush up immediately when the stroke is finished to prevent a paint blob
    printer.move_to(z=up_heigth, feed_rate=FEED_RATE_TRAVEL)

    print(f"--- Stroke {index} execution complete ---")

## Load Data

In [5]:
my_stroke_sequence = StrokeSequence.load_from_json("data/my_stroke_sequence.json")
my_calibration = RobotCalibration.load("data/my_robot_calibration.json")

## Resize to the canvas

In [6]:
my_stroke_sequence.resize_to(my_calibration.get_canvas_size())
print(my_calibration.get_canvas_size())
print(my_stroke_sequence.image_size)
my_stroke_sequence.save_to_json("data/resized_stroke_sequence.json")

(143, 188)
(143, 188)


## Load Brush

In [7]:
color_index = 0 # red
load_brush(printer, my_calibration, color_index)

--- Starting Full Brush Prep Sequence ---
Moving to water reservoir at (186.6, 179.5)...
Agitating brush in water...
Moving to paint palette at (194.5, 42.2)...
Swirling brush to load paint...
--- Brush prep complete and ready to paint! ---


## Execute Stroke

In [8]:
execute_stroke(
    printer=printer,
    robot_calibration=my_calibration,
    stroke_sequence=my_stroke_sequence,
    index=0
)

--- Executing Stroke 0 | Color: (179, 205, 222) | Width: 11 ---
Path [(100, 46), (101, 45), (103, 45), (105, 45), (106, 46), (108, 46), (110, 46)]
--- Stroke 0 execution complete ---
